# Experiments
Notebook per confrontare run con iperparametri diversi.
Il dataset viene costruito e splittato **una volta sola** e condiviso tra tutti gli esperimenti per un confronto equo.
I log vengono salvati in `experiment_logs/<nome>/` come CSV di PyTorch Lightning.

In [ ]:
import sys, os

import yaml
sys.path.insert(0, 'src')

import torch
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning.loggers import CSVLogger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, TQDMProgressBar

from src.complex_dataset import ComplexDataset, complex_collate_fn, build_complex_dataset
from src.baseline import BaselineGNN
from src.graphmatchingnetwork import GraphMatchingNetwork

#per migliorare le prestazioni su GPU 
torch.set_float32_matmul_precision('medium')

config_path = 'config.yaml'
with open(config_path) as f:
        config = yaml.safe_load(f)

PROCESSED_DIR = config['data']['processed_dir']
CSV_PATH      = config['data']['csv_path']
LOGS_DIR      = 'experiment_logs'
SEED          = config['data']['seed']

torch.manual_seed(SEED)

# experiment analysis


In [ ]:
from matplotlib.ticker import MaxNLocator
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import glob

LIGHTNING_LOGS_DIR = 'lightning_logs'

def _scan_experiments(logs_dir):
    entries = []
    for exp_path in sorted(glob.glob(os.path.join(logs_dir, '*'))):
        if not os.path.isdir(exp_path):
            continue
        exp_name = os.path.basename(exp_path)
        for ver_path in sorted(glob.glob(os.path.join(exp_path, 'version_*'))):
            ver_name = os.path.basename(ver_path)
            entries.append((f"{exp_name} / {ver_name}", exp_path, ver_path))
    return entries

def _load_metrics(ver_path):
    csv_path = os.path.join(ver_path, 'metrics.csv')
    return pd.read_csv(csv_path) if os.path.exists(csv_path) else None

def _load_hparams(ver_path):
    hp_path = os.path.join(ver_path, 'hparams.yaml')
    if not os.path.exists(hp_path):
        return {}
    with open(hp_path) as f:
        return yaml.safe_load(f) or {}

def _render_dashboard(ver_path, out):
    with out:
        clear_output(wait=True)
        metrics = _load_metrics(ver_path)
        hparams = _load_hparams(ver_path)

        if metrics is None:
            print("metrics.csv non trovato in", ver_path)
            return

        val_col  = metrics['val_loss'].dropna()
        pear_col = metrics['val_pearson_r'].dropna() if 'val_pearson_r' in metrics else pd.Series(dtype=float)

        # LR: cerca la prima colonna che inizia con 'lr-' (lr-AdamW, lr-Adam, ecc.)
        lr_col_name = next((c for c in metrics.columns if c.startswith('lr-')), None)
        lr_data = metrics[['step', lr_col_name]].dropna() if lr_col_name else None

        best_val  = val_col.min()     if len(val_col)  else float('nan')
        best_pear = pear_col.max()    if len(pear_col) else float('nan')
        last_pear = pear_col.iloc[-1] if len(pear_col) else float('nan')
        tot_epochs = int(metrics['epoch'].max()) + 1

        # --- riepilogo testo ---
        print("─" * 60)
        print("  Iperparametri")
        print("─" * 60)
        for k, v in hparams.items():
            print(f"  {k:<28} {v}")
        print()
        print("─" * 60)
        print("  Riepilogo metriche")
        print("─" * 60)
        print(f"  Epoche totali:          {tot_epochs}")
        print(f"  Best val_loss (MSE):    {best_val:.4f}")
        if not pd.isna(best_pear):
            print(f"  Best val_pearson_r:     {best_pear:.4f}   (finale: {last_pear:.4f})")
        print()

        train_m = metrics[metrics['train_loss'].notna()]
        train_per_epoch = train_m.groupby('epoch')['train_loss'].mean()
        val_m = metrics[metrics['val_loss'].notna()].set_index('epoch')['val_loss']

        has_lr   = lr_data is not None and len(lr_data) > 0
        has_pear = len(pear_col) > 0

        # --- riga superiore: fino a 3 pannelli ---
        n_top = 1 + int(has_lr) + int(has_pear)
        fig_top, axes = plt.subplots(1, n_top, figsize=(5 * n_top, 4))
        axes = list(axes) if n_top > 1 else [axes]

        axes[0].plot(train_per_epoch.index, train_per_epoch.values,
                     color='steelblue', linewidth=1.5, label='train loss')
        axes[0].plot(val_m.index, val_m.values,
                     color='tomato', marker='o', markersize=2, linewidth=1.5, label='val loss')
        axes[0].set_title('Loss (per epoch)')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('MSE Loss')
        axes[0].legend()

        ax_idx = 1

        if has_lr:
            axes[ax_idx].plot(lr_data['step'], lr_data[lr_col_name],
                              color='darkorange', linewidth=1.5)
            axes[ax_idx].set_title(f'Learning Rate ({lr_col_name})')
            axes[ax_idx].set_xlabel('Step')
            axes[ax_idx].set_ylabel('LR')
            axes[ax_idx].ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
            ax_idx += 1

        if has_pear:
            pear_m = metrics[metrics['val_pearson_r'].notna()]
            axes[ax_idx].plot(pear_m['epoch'], pear_m['val_pearson_r'],
                              color='darkorchid', marker='^', markersize=3, linewidth=1.5)
            axes[ax_idx].set_title('Val Pearson r (per epoch)')
            axes[ax_idx].set_xlabel('Epoch')
            axes[ax_idx].set_ylabel('Pearson r')

        for ax in axes:
            ax.grid(True, alpha=0.3)
        fig_top.tight_layout()
        plt.show()

        # --- riga inferiore: zoom convergenza ---
        cutoff = max(1, int(len(train_per_epoch) * 0.15))
        all_conv = pd.concat([train_per_epoch.iloc[cutoff:], val_m.iloc[cutoff:]])
        y_min = all_conv.min() * 0.97
        y_max = np.percentile(all_conv, 97) * 1.03

        fig_zoom, ax_z = plt.subplots(figsize=(5 * n_top, 4))
        ax_z.plot(train_per_epoch.index, train_per_epoch.values,
                  color='steelblue', linewidth=0.9, alpha=0.85, label='train loss')
        ax_z.plot(val_m.index, val_m.values,
                  color='tomato', linewidth=0.9, marker='o', markersize=1.5, alpha=0.85, label='val loss')
        ax_z.set_ylim(y_min, y_max)
        ax_z.yaxis.set_major_locator(MaxNLocator(nbins=12))
        ax_z.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=20))
        ax_z.set_title('Loss — zoom sulla convergenza')
        ax_z.set_xlabel('Epoch')
        ax_z.set_ylabel('MSE Loss')
        ax_z.legend()
        ax_z.grid(True, alpha=0.35, linewidth=0.6)
        fig_zoom.tight_layout()
        plt.show()

# --- UI ---
experiments = _scan_experiments(LIGHTNING_LOGS_DIR)

if not experiments:
    print(f"Nessun esperimento trovato in '{LIGHTNING_LOGS_DIR}/'.")
else:
    labels = [e[0] for e in experiments]
    dd = widgets.Dropdown(
        options=labels,
        description='Esperimento:',
        layout=widgets.Layout(width='420px'),
        style={'description_width': 'initial'},
    )
    out = widgets.Output()

    def _on_change(change):
        if change['name'] == 'value':
            idx = labels.index(change['new'])
            _render_dashboard(experiments[idx][2], out)

    dd.observe(_on_change, names='value')
    display(dd, out)
    _render_dashboard(experiments[0][2], out)

## Dataset — costruito una volta sola

In [ ]:
from sklearn.model_selection import KFold

N_FOLDS = 5
full_dataset = build_complex_dataset(PROCESSED_DIR, CSV_PATH, limit=2000)
all_codes = full_dataset.pdb_codes

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = [(list(tr), list(vl)) for tr, vl in kf.split(all_codes)]

print(f"Dataset: {len(all_codes)} complessi  →  {N_FOLDS}-fold CV")
for i, (tr, vl) in enumerate(splits):
    print(f"  Fold {i}: train={len(tr)}  val={len(vl)}")

## Configurazioni esperimenti
Aggiungi o modifica entry nella lista `EXPERIMENTS`. Ogni dict deve avere il campo `name` (usato come nome della cartella log).

In [ ]:
# Iperparametri condivisi tra tutte le coppie
EPOCHS     = 400
PATIENCE   = 15
TOKEN_DIM  = 32
NODE_DIMS  = [32,16]
EDGE_DIMS  = [32,16]
NUM_LAYERS = 1
BATCH_SIZE = 16

EXPERIMENTS = [
    # ── 1. Nessuna regolarizzazione ────────────────────────
    {
        'name':                 'baseline_no_reg',
        'model':                'baseline',
        'token_dim':            TOKEN_DIM,
        'node_embedding_dims':  NODE_DIMS,
        'edge_embedding_dims':  EDGE_DIMS,
        'num_layers':           NUM_LAYERS,
        'lr':                   1e-3,
        'max_lr':               1e-3,
        'min_lr':               1e-3,
        'warmup_percentage':    0.0,
        'l2_norm':              0.0,
        'dropout_p':            0.0,
        'normalization':        False,
        'use_dynamic_schedule': False,
        'batch_size':           BATCH_SIZE,
    },
    {
        'name':                 'gmn_no_reg',
        'model':                'gmn',
        'token_dim':            TOKEN_DIM,
        'node_embedding_dims':  NODE_DIMS,
        'edge_embedding_dims':  EDGE_DIMS,
        'num_layers':           NUM_LAYERS,
        'lr':                   1e-3,
        'max_lr':               1e-3,
        'min_lr':               1e-3,
        'warmup_percentage':    0.0,
        'l2_norm':              0.0,
        'dropout_p':            0.0,
        'normalization':        False,
        'use_dynamic_schedule': False,
        'batch_size':           BATCH_SIZE,
    },

    # ── 2. Regolarizzazione (norm + dropout + L2) ──────────
    {
        'name':                 'baseline_reg',
        'model':                'baseline',
        'token_dim':            TOKEN_DIM,
        'node_embedding_dims':  NODE_DIMS,
        'edge_embedding_dims':  EDGE_DIMS,
        'num_layers':           NUM_LAYERS,
        'lr':                   1e-3,
        'max_lr':               1e-3,
        'min_lr':               1e-3,
        'warmup_percentage':    0.0,
        'l2_norm':              0.01,
        'dropout_p':            0.1,
        'normalization':        True,
        'use_dynamic_schedule': False,
        'batch_size':           BATCH_SIZE,
    },
    {
        'name':                 'gmn_reg',
        'model':                'gmn',
        'token_dim':            TOKEN_DIM,
        'node_embedding_dims':  NODE_DIMS,
        'edge_embedding_dims':  EDGE_DIMS,
        'num_layers':           NUM_LAYERS,
        'lr':                   1e-3,
        'max_lr':               1e-3,
        'min_lr':               1e-3,
        'warmup_percentage':    0.0,
        'l2_norm':              0.01,
        'dropout_p':            0.1,
        'normalization':        True,
        'use_dynamic_schedule': False,
        'batch_size':           BATCH_SIZE,
    },

    # ── 3. Regolarizzazione + schedule dinamico ────────────
    {
        'name':                 'baseline_dyn_sched',
        'model':                'baseline',
        'token_dim':            64,
        'node_embedding_dims':  NODE_DIMS,
        'edge_embedding_dims':  EDGE_DIMS,
        'num_layers':           2,
        'lr':                   1e-3,
        'max_lr':               1e-3,
        'min_lr':               1e-6,
        'warmup_percentage':    0.075,
        'l2_norm':              0.0006,
        'dropout_p':            0.12,
        'normalization':        True,
        'use_dynamic_schedule': True,
        'batch_size':           BATCH_SIZE,
    },
    {
        'name':                 'gmn_dyn_sched',
        'model':                'gmn',
        'token_dim':            64,
        'node_embedding_dims':  NODE_DIMS,
        'edge_embedding_dims':  EDGE_DIMS,
        'num_layers':           2,
        'lr':                   1e-3,
        'max_lr':               1e-3,
        'min_lr':               1e-6,
        'warmup_percentage':    0.075,
        'l2_norm':              0.0006,
        'dropout_p':            0.12,
        'normalization':        True,
        'use_dynamic_schedule': True,
        'batch_size':           BATCH_SIZE,
    },
]

## Funzione di training

In [ ]:
from pytorch_lightning.callbacks import LearningRateMonitor

def run_experiment(cfg, train_codes, val_codes, fold_idx=None):
    train_ds = ComplexDataset(PROCESSED_DIR, train_codes)
    val_ds   = ComplexDataset(PROCESSED_DIR, val_codes)

    train_loader = DataLoader(
        train_ds, batch_size=cfg['batch_size'], shuffle=True,
        collate_fn=complex_collate_fn, num_workers=4,
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg['batch_size'], shuffle=False,
        collate_fn=complex_collate_fn, num_workers=4,
    )

    model_kwargs = dict(
        token_dim=cfg['token_dim'],
        node_embedding_dims=cfg['node_embedding_dims'],
        edge_embedding_dims=cfg['edge_embedding_dims'],
        num_layers=cfg['num_layers'],
        lr=cfg['lr'],
        max_lr=cfg['max_lr'],
        min_lr=cfg['min_lr'],
        warmup_step_percentage=cfg['warmup_percentage'],
        l2_norm=cfg['l2_norm'],
        dropout_p=cfg['dropout_p'],
        normalization=cfg['normalization'],
        use_dynamic_schedule=cfg['use_dynamic_schedule'],
    )

    model = GraphMatchingNetwork(**model_kwargs) if cfg['model'] == 'gmn' else BaselineGNN(**model_kwargs)

    version = f"fold_{fold_idx}" if fold_idx is not None else None
    logger = CSVLogger(LOGS_DIR, name=cfg['name'], version=version)

    callbacks = [
        ModelCheckpoint(dirpath=logger.log_dir, filename='best',
                        monitor='val_loss', mode='min', save_top_k=0),
        EarlyStopping(monitor='val_loss', patience=PATIENCE, mode='min'),
        LearningRateMonitor(logging_interval='step'),
        TQDMProgressBar(refresh_rate=10),
    ]

    trainer = pl.Trainer(
        max_epochs=EPOCHS,
        precision='bf16-mixed',
        log_every_n_steps=10,
        logger=logger,
        callbacks=callbacks,
        enable_progress_bar=True,
    )

    trainer.fit(model, train_loader, val_loader)
    return pd.read_csv(os.path.join(logger.log_dir, 'metrics.csv'))

## Run

In [ ]:
import json

cv_results = {}  # name -> list of {val_loss, pearson_r} per fold

for cfg in EXPERIMENTS:
    name = cfg['name']
    print(f"\n{'='*55}\n  {name}  ({N_FOLDS}-fold CV)\n{'='*55}")
    fold_scores = []

    for fold_idx, (train_idx, val_idx) in enumerate(splits):
        train_codes = [all_codes[i] for i in train_idx]
        val_codes   = [all_codes[i] for i in val_idx]

        print(f"\n  -- Fold {fold_idx + 1}/{N_FOLDS} --")
        metrics = run_experiment(cfg, train_codes, val_codes, fold_idx=fold_idx)

        best_val = float(metrics['val_loss'].dropna().min())
        best_r   = float(metrics['val_pearson_r'].dropna().max()) if 'val_pearson_r' in metrics else float('nan')
        fold_scores.append({'val_loss': best_val, 'pearson_r': best_r})
        print(f"  Fold {fold_idx + 1}:  val_loss={best_val:.4f}  pearson_r={best_r:.4f}")

    cv_results[name] = fold_scores
    vl_vals = [f['val_loss'] for f in fold_scores]
    r_vals  = [f['pearson_r'] for f in fold_scores]
    print(f"\n  >> val_loss  {np.mean(vl_vals):.4f} ± {np.std(vl_vals):.4f}")
    print(f"  >> pearson_r {np.mean(r_vals):.4f} ± {np.std(r_vals):.4f}")

os.makedirs(LOGS_DIR, exist_ok=True)
cv_path = os.path.join(LOGS_DIR, 'cv_results.json')
with open(cv_path, 'w') as f:
    json.dump(cv_results, f, indent=2)
print(f"\nRisultati salvati in {cv_path}")

## Risultati

In [ ]:
import os
import pandas as pd
import numpy as np

# Imposta il numero corretto di fold
N_FOLDS = 5 

summary = []

for cfg in EXPERIMENTS:
    name = cfg['name']
    
    vl_list = []
    r_list = []
    mae_list = []
    
    for fold_idx in range(N_FOLDS):
        csv_path = os.path.join(LOGS_DIR, name, f'fold_{fold_idx}', 'metrics.csv')
        
        try:
            df_metrics = pd.read_csv(csv_path)
            
            # Controlla se val_loss esiste e non è completamente vuota
            if 'val_loss' in df_metrics.columns and not df_metrics['val_loss'].dropna().empty:
                
                # TROVA L'EPOCA (INDICE) CON IL MINIMO MSE
                best_epoch_idx = df_metrics['val_loss'].idxmin()
                
                # 1. Estrai la val_loss minima
                vl_list.append(float(df_metrics.loc[best_epoch_idx, 'val_loss']))
                
                # 2. Estrai Pearson r in QUELLA specifica epoca
                if 'val_pearson_r' in df_metrics.columns and not pd.isna(df_metrics.loc[best_epoch_idx, 'val_pearson_r']):
                    r_list.append(float(df_metrics.loc[best_epoch_idx, 'val_pearson_r']))
                else:
                    r_list.append(float('nan'))
                    
                # 3. Estrai MAE in QUELLA specifica epoca
                if 'val_mae' in df_metrics.columns and not pd.isna(df_metrics.loc[best_epoch_idx, 'val_mae']):
                    mae_list.append(float(df_metrics.loc[best_epoch_idx, 'val_mae']))
                else:
                    mae_list.append(float('nan'))
                    
            else:
                # Se non c'è val_loss, riempiamo con NaN
                vl_list.append(float('nan'))
                r_list.append(float('nan'))
                mae_list.append(float('nan'))
                
        except (FileNotFoundError, KeyError):
            vl_list.append(float('nan'))
            r_list.append(float('nan'))
            mae_list.append(float('nan'))

    # Salta gli esperimenti per cui non abbiamo trovato alcun CSV valido
    if np.all(np.isnan(vl_list)):
        print(f"Nessun file CSV valido trovato per: {name}. Salto...")
        continue

    # Compila il dizionario riassuntivo
    summary.append({
        'name':                name,
        'model':               cfg['model'],
        'val_loss_mean':       round(np.nanmean(vl_list), 4),
        'val_loss_std':        round(np.nanstd(vl_list),  4),
        'val_mae_mean':        round(np.nanmean(mae_list), 4),
        'val_mae_std':         round(np.nanstd(mae_list),  4),
        'pearson_r_mean':      round(np.nanmean(r_list),  4),
        'pearson_r_std':       round(np.nanstd(r_list),   4),
        'node_embedding_dims': cfg['node_embedding_dims'][-1],
        'edge_embedding_dims': cfg['edge_embedding_dims'][-1],
        'num_layers':          cfg['num_layers'],
        'normalization':       cfg['normalization'],
        'dropout_p':           cfg['dropout_p'],
        'l2_norm':             cfg['l2_norm'],
        'use_dynamic_schedule':cfg['use_dynamic_schedule'],
        'max_lr':              cfg['max_lr'],
        'batch_size':          cfg['batch_size'],
    })

# Creazione e salvataggio
df_summary = pd.DataFrame(summary).set_index('name').sort_values('val_loss_mean')

summary_path = os.path.join(LOGS_DIR, 'cv_summary.csv')
df_summary.to_csv(summary_path)
print(f"Summary salvata in {summary_path}\n")

# Stampa a video il DataFrame finale
df_summary

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# 1. Definiamo le categorie per allinearci ai nomi nel DataFrame
configs = ['no_reg', 'reg', 'dyn_sched']
models = ['baseline', 'gmn']

# 2. Inizializziamo i dizionari per raccogliere i dati (rimosso MAE)
vl_means, vl_stds = {m: [] for m in models}, {m: [] for m in models}
r_means,  r_stds  = {m: [] for m in models}, {m: [] for m in models}

# 3. Estraiamo i dati direttamente da df_summary
for m in models:
    for c in configs:
        name = f"{m}_{c}"
        
        # Controllo di sicurezza
        if name in df_summary.index:
            vl_means[m].append(df_summary.loc[name, 'val_loss_mean'])
            vl_stds[m].append(df_summary.loc[name, 'val_loss_std'])
            
            r_means[m].append(df_summary.loc[name, 'pearson_r_mean'])
            r_stds[m].append(df_summary.loc[name, 'pearson_r_std'])
        else:
            vl_means[m].append(np.nan); vl_stds[m].append(np.nan)
            r_means[m].append(np.nan);  r_stds[m].append(np.nan)

# ==========================================
# 4. PLOTTING (2 Subplots)
# ==========================================
x = np.arange(len(configs))
width = 0.35

# Torniamo a 2 colonne e accorciamo la larghezza (12 invece di 18)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

err_color = 'gray'
err_style = {'elinewidth': 1.2, 'capthick': 1.2, 'alpha': 0.6}
xtick_labels = ["Standard", "Regularized", "Dynamic-Large"]

# --- GRAFICO 1: Val Loss MSE ---
axes[0].bar(x - width/2, vl_means['baseline'], width, yerr=vl_stds['baseline'], 
            label='Baseline', color='#1f77b4', capsize=5, alpha=0.85, ecolor=err_color, error_kw=err_style)
axes[0].bar(x + width/2, vl_means['gmn'], width, yerr=vl_stds['gmn'], 
            label='GMN', color='#ff7f0e', capsize=5, alpha=0.85, ecolor=err_color, error_kw=err_style)

axes[0].set_title('Val Loss MSE')
axes[0].set_ylabel('MSE')
axes[0].set_xticks(x)
axes[0].set_xticklabels(xtick_labels, rotation=30, ha='right', fontsize=9)
axes[0].legend()
axes[0].grid(True, axis='y', alpha=0.3)

all_vl_min = np.nanmin(np.array(vl_means['baseline'] + vl_means['gmn']) - np.array(vl_stds['baseline'] + vl_stds['gmn']))
all_vl_max = np.nanmax(np.array(vl_means['baseline'] + vl_means['gmn']) + np.array(vl_stds['baseline'] + vl_stds['gmn']))
axes[0].set_ylim([all_vl_min * 0.95, all_vl_max * 1.05])

# --- GRAFICO 2: Val Pearson r ---
axes[1].bar(x - width/2, r_means['baseline'], width, yerr=r_stds['baseline'], 
            label='Baseline', color='#9467bd', capsize=5, alpha=0.85, ecolor=err_color, error_kw=err_style)
axes[1].bar(x + width/2, r_means['gmn'], width, yerr=r_stds['gmn'], 
            label='GMN', color='#2ca02c', capsize=5, alpha=0.85, ecolor=err_color, error_kw=err_style)

axes[1].set_title('Val Pearson r')
axes[1].set_ylabel('Pearson r')
axes[1].set_xticks(x)
axes[1].set_xticklabels(xtick_labels, rotation=30, ha='right', fontsize=9)
axes[1].legend()
axes[1].grid(True, axis='y', alpha=0.3)

all_r_min = np.nanmin(np.array(r_means['baseline'] + r_means['gmn']) - np.array(r_stds['baseline'] + r_stds['gmn']))
all_r_max = np.nanmax(np.array(r_means['baseline'] + r_means['gmn']) + np.array(r_stds['baseline'] + r_stds['gmn']))
axes[1].set_ylim([all_r_min * 0.95, all_r_max * 1.05]) 

# ==========================================
# 5. Salvataggio
# ==========================================
# Assicurati di avere la variabile N_FOLDS definita prima nel notebook se vuoi includerla nel titolo
plt.suptitle('Cross-Validation Results (mean ± std)', fontsize=14, y=1.05)
plt.tight_layout()

os.makedirs(LOGS_DIR, exist_ok=True)
plt.savefig(os.path.join(LOGS_DIR, 'cv_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

# LR tuner

In [ ]:
from pytorch_lightning.tuner import Tuner
from src.utils import ComplexDataModule

data_module = ComplexDataModule(
    processed_dir=PROCESSED_DIR,
    csv_path=CSV_PATH,
    batch_size=16,
    limit=1000,
    train_val_split=0.2,
    seed=SEED,
)

model = GraphMatchingNetwork(
    token_dim=16,
    node_embedding_dims=[48],
    edge_embedding_dims=[32],
    num_layers=1,
    lr=0.001,
    dropout_p=0.0,
    normalization=True,
)

trainer = pl.Trainer(
    max_epochs=config['training']['epochs'],
    precision='bf16-mixed',
)

print("Avvio la scansione del Learning Rate...")
tuner = Tuner(trainer)
lr_finder = tuner.lr_find(
    model, 
    datamodule=data_module,
    num_training=200, 
    min_lr=1e-8, 
    max_lr=1
)


In [ ]:
# stampa il learning rate suggerito
new_lr = lr_finder.suggestion()
print(f"LR suggerito: {new_lr:.2e}")
fig = lr_finder.plot(suggest=True)
fig.show()